## Creating rolling loops

In [1]:
import os

os.chdir("C:/Users/finle/Dev/Dissertation")

In [2]:

from pandas import Series
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.models import rolling_garch_var
spy = pd.read_csv("data/SPY_data.csv", index_col=0, parse_dates=True)

log_returns = spy['log_returns'].fillna(0)
type(log_returns)
spy.index
type(log_returns.index)


pandas.core.indexes.datetimes.DatetimeIndex

Historical VaR


In [ ]:
def VaR_historical_rolling(
    ts: Series, 
    window: int,
    confidence: float =0.99
    ):
    if confidence >= 0.9999:
        raise ValueError(f"Confidence must be below 1. you entered {confidence}")
    VaR = - ts.rolling(window,min_periods=50).quantile(1-confidence).shift(1)
    return VaR


historical_rolling_95 = VaR_historical_rolling(log_returns, 500, 0.95)
historical_rolling_99 = VaR_historical_rolling(log_returns, 500, 0.99)

rolling_forecasts =  pd.DataFrame({'historical_rolling_95': historical_rolling_95,
             'historical_rolling_99':historical_rolling_99})

ewma

In [ ]:
from scipy.stats import norm

def ewma_rolling_var(
    returns:Series,
    lam: float = 0.94,
    alpha: float = 0.01,
    confidence: float = 0.99):
    """
    Parameters
    ----------
    lam: float
        Decay parameter. Riskmetrics uses 0.94 for daily data.
    alpha: float   
        Tail probability.
        
    Returns
    -------
    """
    r = np.array(returns)
    T = len(r)
    var = np.zeros(T)
    var[0] = np.var(r[:30])
    
    for t in range(1,T):
        var[t] = lam * var[t-1] + (1 - lam) * r[t-1] ** 2
        
    sigma = np.sqrt(var)
    VaR = -(np.mean(r) + sigma * norm.ppf(1 - confidence))
    return VaR

ewma_rolling_95 = ewma_rolling_var(log_returns, confidence=0.95)
ewma_rolling_99 = ewma_rolling_var(log_returns, confidence=0.99)

rolling_forecasts['ewma_rolling_95'] = ewma_rolling_95
rolling_forecasts['ewma_rolling_99'] = ewma_rolling_99
rolling_forecasts.to_csv("data/historical_rolling_forecasts.csv")

GARCH Rolling VaR

In [ ]:
garch_rolling_95_t = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.05,
    horizon=1,
    dist='t')

garch_rolling_99_t = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.01,
    horizon=1,
    dist='t')




normal based VaR

In [ ]:
garch_rolling_95_n = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.05,
    horizon=1,
    dist='normal')

garch_rolling_99_n = rolling_garch_var(
    log_returns,
    window=500,
    alpha=0.01,
    horizon=1,
    dist='normal')


garch_results = pd.DataFrame({
    'returns': np.ravel(garch_rolling_95_t[2]),
    'garch_rolling_95_t':np.ravel(garch_rolling_95_t[1]),
    'garch_rolling_99_t':np.ravel(garch_rolling_99_t[1]),
    'garch_rolling_95_n':np.ravel(garch_rolling_95_n[1]),
    'garch_rolling_99_n':np.ravel(garch_rolling_99_n[1])
},
                             index=np.ravel(garch_rolling_95_t[0])
                             )



In [ ]:
garch_results.to_csv("data/garch_rolling_forecasts.csv")

MS model

In [ ]:
from src.models import rolling_ms_var

MS_rolling_var_95 = rolling_ms_var(log_returns, 2, window=500, alpha=0.05)

In [ ]:
fig, ax = plt.subplots(figsize=(12,5), dpi=600)

ax.plot(MS_rolling_var_95[0], -MS_rolling_var_95[1])
ax.plot(MS_rolling_var_95[0],MS_rolling_var_95[2])
ax.grid(True)


In [ ]:
MS_rolling_var_99 = rolling_ms_var(
    log_returns,
    k_regimes=2,
    window=500,
    alpha=0.01,
    horizon=1)

In [ ]:
fig, ax = plt.subplots(figsize=(12,5), dpi=600)

ax.plot(MS_rolling_var_99[0], -MS_rolling_var_99[1])
ax.plot(MS_rolling_var_99[0],MS_rolling_var_99[2])
ax.grid(True)

MS-GARCH Model

In [3]:
from src.models import HaasMSGarch

haas = HaasMSGarch(k_regimes=2, dist='t')
haas.fit(log_returns)
haas.summary()

Fitting Haas Markov_Switching GARCH(1,1) Model: K = 2, dist=t
Parameters: 12 | Observations: 5033


Fitting MS_GARCH.:  16%|█▌        | 156/1000 [03:30<18:58,  1.35s/it]

Status: YLL = -6520.7833
Total function evaluations: 2483
Haas MS-GARCH(1,1)

Regimes: 2

Distribution: t

Sample: 2005-01-03

Log-Lik: -6520.783302035175

AIC: 13065.56660407035

BIC: 13143.851862151667

Regimes: 2

Transition Matrix P[i,j] = P(S_t = j| S_{t-1}=i):
           Regime 0 Regime 1
 Reg[0]:0.9267220.073278
 Reg[1]:0.5000000.500000
--------------------------------------------------

Regime Classification:
Regime      Label     Obs     Share     
--------------------------------------------------
0     low volatility     4883     97.0196701768329%
1     high_vol     150     2.9803298231670974%

GARCH(1,1) Parameters by Regime:
Regime     mu     omega      alpha     beta     persistence     nu
--------------------------------------------------
0        0.1295   0.0073     0.1125   0.8626   0.9751       10.526  
    Unconditional variance = 0.2918 (sigma = 0.5402)
1        -0.5624  0.0115     0.0173   0.9760   0.9933       10.235  
    Unconditional variance = 1.7224 (sigma = 

In [4]:
from src.models import HaasMSGarch

haas = HaasMSGarch(k_regimes=2, dist='normal')
haas.fit(log_returns)
haas.summary()

Fitting Haas Markov_Switching GARCH(1,1) Model: K = 2, dist=normal
Parameters: 10 | Observations: 5033


Fitting MS_GARCH.:   6%|▌         | 59/1000 [01:03<16:55,  1.08s/it]

Status: YLL = -6537.9879
Total function evaluations: 770
Haas MS-GARCH(1,1)

Regimes: 2

Distribution: normal

Sample: 2005-01-03

Log-Lik: -6537.98789481101

AIC: 13095.97578962202

BIC: 13161.213504689784

Regimes: 2

Transition Matrix P[i,j] = P(S_t = j| S_{t-1}=i):
          egime 0egime 1
 Reg[0]:0.9038310.096169
 Reg[1]:0.5000000.500000
--------------------------------------------------

Regime Classification:
Regime      Label     Obs     Share     
--------------------------------------------------
0     low volatility     4792     95.21160341744486%
1     high_vol     241     4.788396582555136%

GARCH(1,1) Parameters by Regime:
Regime     mu     omega      alpha     beta     persistence
--------------------------------------------------
0        0.1301   0.0056     0.0867   0.8786   0.9653      
    Unconditional variance = 0.1619 (sigma = 0.4024)
1        -0.4305  0.1170     0.1419   0.8498   0.9917      
    Unconditional variance = 14.1250 (sigma = 3.7583)


In [5]:
print(haas.filtered_probs_)

              P(S=0)    P(S=1)
Price                         
2005-01-03  0.978900  0.021100
2005-01-04  0.209681  0.790319
2005-01-05  0.848018  0.151982
2005-01-06  0.966667  0.033333
2005-01-07  0.975228  0.024772
...              ...       ...
2024-12-24  0.907170  0.092830
2024-12-26  0.915102  0.084898
2024-12-27  0.810353  0.189647
2024-12-30  0.764819  0.235181
2024-12-31  0.837288  0.162712

[5033 rows x 2 columns]
